# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Access as object, not dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their IDs, and preview first few records.

In [ ]:
# List available record sets (@id) in the dataset
record_sets = []
if hasattr(metadata, 'recordSet'):
    rs = metadata.recordSet
    if isinstance(rs, list):
        record_sets = [r['@id'] if isinstance(r, dict) and '@id' in r else r for r in rs]
    elif isinstance(rs, dict):
        record_sets = [rs['@id']] if '@id' in rs else []

print("Available Record Sets (by @id):")
for rs_id in record_sets:
    print(f"- {rs_id}")

if len(record_sets) > 0:
    print("\nPreviewing first 3 records from the first record set:")
    rs_id = record_sets[0]
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No record sets found in dataset metadata.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record set `@id`s are referenced dynamically.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show available columns for each extracted record set
for rs_id, df in dataframes.items():
    print(f"\nRecord Set: {rs_id}")
    print("Fields (@id):", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations can include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Choose the first record set for demonstration
if len(record_sets) > 0:
    rs_id = record_sets[0]
    df = dataframes[rs_id]
    print(f"Working with Record Set: {rs_id}")
    
    # Identify possible numeric fields (IDs with numeric values)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Use first numeric field
        print(f"Numeric field chosen for analysis (@id): {numeric_field}")
        
        # Set threshold for filtering
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())
        
        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Group by another field (e.g., demographic)
        possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No categorical grouping field found.")
    else:
        print("No numeric fields detected in the record set.")
else:
    print("No record sets/data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below is an example visualization for the numeric field selected.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution for the numeric field
if len(record_sets) > 0 and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field is available, show boxplot
    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarized key findings:
- The dataset comprises multiple record sets (see IDs above).
- Exploratory analyses reveal numeric and demographic correlations.
- Visualizations indicate the distribution and grouping behavior of selected numeric indicators.

Further steps can include advanced statistical analysis and machine learning modeling tailored to the available fields and the survey's structure.